# NIH metrics

This notebook uses information extracted from [NIH Exporter](https://reporter.nih.gov/exporter) to identify NIH-NHLBI funded users of PhysioNet.

## Import packages

In [1]:
import os
from pathlib import Path

import pandas as pd

from twentyfiveyears.nih import (combine_exporter_tables, get_physionet_users, get_investigators, get_authors, link_users)

## Setup

In [2]:
# Set the base path
base_path = os.path.join("..", "data")

## Load map of Person IDs

All users are assigned a unique `person_id`.

In [3]:
# Load the map of Person IDs
path = os.path.join(base_path, 'handcrafted', 'person_id_lookup.csv')
person_map = pd.read_csv(path)
person_map.head(3)

,person_id,physionet_id
0,100000000,2
1,100000001,6
2,100000002,8


## Load PhysioNet dataset

Load a dataset containing the list of PhysioNet users

In [4]:
# Load DataFrame of PhysioNet users
path = os.path.join(base_path, 'physionet', 'users.csv')
df_physionet_users = get_physionet_users(path, person_map, first_name_as_initial=True)
df_physionet_users = df_physionet_users[0:9]
df_physionet_users.head(3)

,person_id,physionet_name
0,100000000,f torres fábregas
1,100000001,t pollard
2,100000002,b moody


## Load Principal Investigators of NIH projects

Load a list of Principal Investigators

In [5]:
# Load the NIH project data
path = os.path.join(base_path, 'nih', 'exporter', 'projects')
df_projects = combine_exporter_tables(path, "RePORTER_PRJ_C_FY", start_year=1995)

### Limit the data to NIBIB projects

In [6]:
df_projects = df_projects[df_projects['IC_NAME']=='National Heart, Lung, and Blood Institute'.upper()]
df_projects.head(3)

,APPLICATION_ID,ACTIVITY,ADMINISTERING_IC,APPLICATION_TYPE,ARRA_FUNDED,AWARD_NOTICE_DATE,BUDGET_START,BUDGET_END,CFDA_CODE,CORE_PROJECT_NUM,...,SUBPROJECT_ID,SUFFIX,SUPPORT_YEAR,TOTAL_COST,TOTAL_COST_SUB_PROJECT,OPPORTUNITY NUMBER,FUNDING_MECHANISM,ORG_IPF_CODE,DIRECT_COST_AMT,INDIRECT_COST_AMT
1995,2213630,F31,HL,5,NaN,1994-12-16T00:00:00,01/20/1995,01/19/1996,837,F31HL008814,...,NaN,NaN,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1996,2213634,F31,HL,5,NaN,1995-08-20T00:00:00,08/31/1995,08/30/1996,837,F31HL008815,...,NaN,NaN,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1997,2213969,F31,HL,5,NaN,1994-11-30T00:00:00,11/01/1994,10/31/1995,838,F31HL009056,...,NaN,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Get the names of Principal Investigators
investigators = get_investigators(df_projects, first_name_as_initial=True)
investigators[0:3]

## Load authors of publications linked to NIH projects

Load a list of authors linked to NIH projects

In [15]:
# Import the linking tables to connect publications / authors to a specific NIH institute
# NOTE: have to manually rename years 2016 - 2020 as RePORTER instead of REPORTER
path = os.path.join(base_path, 'nih', 'exporter', 'link_tables')
df_links = combine_exporter_tables(path, "RePORTER_PUBLNK_C_", start_year=1995)

In [16]:
# Load the NIH publications data
path = os.path.join(base_path, 'nih', 'exporter', 'publications')
df_publications = combine_exporter_tables(path, "RePORTER_PUB_C_", start_year=1995)

### Limit the data to NIBIB publications 

In [17]:
# First get the PROJECT_NUM from the df_links table
df_publications = pd.merge(df_publications, df_links, on='PMID')
# Next get merge with the projects DataFrame to get 'IC_NAME'
df_publications = pd.merge(df_publications, df_projects[['CORE_PROJECT_NUM', 'IC_NAME']].copy(), left_on='PROJECT_NUMBER', right_on='CORE_PROJECT_NUM')

In [19]:
# Get a DataFrame only for NIBIB publications
df_publications = df_publications[df_publications['IC_NAME']=='National Heart, Lung, and Blood Institute'.upper()]
df_publications.head(3)

,AFFILIATION,AUTHOR_LIST,COUNTRY,ISSN,JOURNAL_ISSUE,JOURNAL_TITLE,JOURNAL_TITLE_ABBR,JOURNAL_VOLUME,LANG,PAGE_NUMBER,PMC_ID,PMID,PUB_DATE,PUB_TITLE,PUB_YEAR,PROJECT_NUMBER,CORE_PROJECT_NUM,IC_NAME
0,"Brookhaven National Laboratory, Upton, NY 1197...","Pullia, A; Kraner, H W; Siddons, D P; Furenlid...",United States,0018-9499,4,IEEE transactions on nuclear science,IEEE Trans Nucl Sci,42,eng,585-589,4629789,26538683,1995 Aug,Silicon Detector System for High Rate EXAFS Ap...,1995,P41EB002035,P41EB002035,NATIONAL INSTITUTE OF BIOMEDICAL IMAGING AND B...
1,"Brookhaven National Laboratory, Upton, NY 1197...","Pullia, A; Kraner, H W; Siddons, D P; Furenlid...",United States,0018-9499,4,IEEE transactions on nuclear science,IEEE Trans Nucl Sci,42,eng,585-589,4629789,26538683,1995 Aug,Silicon Detector System for High Rate EXAFS Ap...,1995,P41EB002035,P41EB002035,NATIONAL INSTITUTE OF BIOMEDICAL IMAGING AND B...
2,"Brookhaven National Laboratory, Upton, NY 1197...","Pullia, A; Kraner, H W; Siddons, D P; Furenlid...",United States,0018-9499,4,IEEE transactions on nuclear science,IEEE Trans Nucl Sci,42,eng,585-589,4629789,26538683,1995 Aug,Silicon Detector System for High Rate EXAFS Ap...,1995,P41EB002035,P41EB002035,NATIONAL INSTITUTE OF BIOMEDICAL IMAGING AND B...


In [20]:
# Get the names of authors
authors = get_authors(df_publications, first_name_as_initial=True)
authors[0:3]

['a pullia', 'h kraner', 'd siddons']

## Match NIH listed people to PhysioNet users

Attempt to match people between the two sources

In [21]:
# Match NIH Principal Investigators to PhysioNet users
# Set limit for testing
limit = None
df_physionet_users = link_users(df_physionet_users, investigators, match_group="investigators", limit=limit)

100%|██████████| 9/9 [00:00<00:00, 53.92it/s]


Finished in 1.2405900955200195 seconds


In [22]:
# Match NIH authors to PhysioNet users
df_physionet_users = link_users(df_physionet_users, authors, match_group="authors", limit=limit)

100%|██████████| 9/9 [00:00<00:00, 5618.21it/s]

Finished in 0.17103314399719238 seconds


In [23]:
df_physionet_users.head(5)

,person_id,physionet_name,matched_investigator_score,matched_investigator_name,matched_author_score,matched_author_name
0,100000000,f torres fábregas,0.894118,f torres,0.894118,f torres
1,100000001,t pollard,0.844444,t moller,1.000000,t pollard
2,100000002,b moody,0.849206,n mody,1.000000,b moody
3,100000003,a johnson,0.925926,g johnson,1.000000,a johnson
4,100000004,j ishii-rousseau,0.800000,j hu,0.846667,j rousseau


Save the results

## Save the results

In [24]:
# Save the results
save_path = os.path.join(base_path, 'physionet_users_nih_nhlbi_funded.csv')
path = Path(save_path)

# Convert to a path that works on the current OS
normalized_path = path.as_posix() if path.drive else Path(*path.parts).resolve()

# Output the merged DataFrame or save it to a file
df_physionet_users.to_csv(normalized_path, index=False)